# What this file does
- Influencial complaint analysis based on louvain community (WIP)

# Dependencies
#### Run the following file(s) before running this code.
- 03_baseline_similarity_graph.ipynb (or 03b or 03c)
- 07_gds_Louvain_Summary.ipynb
- 21_gds_Centrality_on-graph.ipynb
- 23_gds_Company.ipynb

##### Note:
URL of the Neo4j browser:
- https://[IP address]:7473/browser/

ID & Pass: 
- Use the one in .env


In [1]:
# Config
SAMPLING:bool       = True
NUM_SAMPLE:int      = 250   # Number of sample data to be ingested to the graph database
SUMMARY_SAMPLE:int  = 20    # Number of samples as inputs of summarizing
RAND_SEED:int       = 77    # Seed for sampling
NUM_SIM:int         = 3     # Number of results from KNN search (does not include the own node)
EMBEDDING_MODEL:str = "text-embedding-3-small"
MAX_TOKENS:int      = 7800  # Max 8192 - some safety buffer about 5%
INDEX_NAME:str      = "idx:complaints_vss"
FILE_PATH:str       = "../data/original/complaints-2025-11-02_04_18.csv"

In [2]:
import time
from datetime import datetime, timedelta

In [3]:
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display
import tiktoken
import textwrap
import logging

logger = logging.getLogger("neo4j")
logger.setLevel(logging.CRITICAL)

In [4]:
from dotenv import load_dotenv  
load_dotenv()

True

In [5]:
import neo4j

In [6]:
# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

In [7]:
warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [8]:
# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows
pd.set_option('display.max_rows', None)

In [9]:
# Show full column
pd.set_option('display.max_colwidth', None)

In [10]:
# Adjust the alignment in jupyter
from IPython.display import HTML

HTML("""
<style>
table.dataframe th {
    text-align: center !important;
    vertical-align: middle;
}
table.dataframe td {
    text-align: left !important;
    vertical-align: top !important;
}
</style>
""")

In [11]:
# Timestamp (Start)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

start_time = time.time()

2026-01-08_22:23:58


### Neo4j

In [12]:
driver = neo4j.GraphDatabase.driver(
    uri=os.environ.get("NEO4J_URI"), 
    auth=(os.environ.get("NEO4J_USERNAME"), 
          os.environ.get("NEO4J_PASSWORD"))
)

In [13]:
session = driver.session(database="neo4j")

In [14]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

# Influential Complaint analysis based on Corporate community

In [15]:
# Get all corporate community ids of Complaint nodes
query = """
MATCH (comp:Complaint)
RETURN comp.corp_community_id
"""

corp_community_ids = my_neo4j_run_query_pandas(query)["comp.corp_community_id"]

In [16]:
# Remove duplicates
corp_community_ids = set(corp_community_ids)

#corp_community_ids

In [17]:
# for id in corp_community_ids:
#     print(id)

In [18]:
###
### Run PageRank for each community
###


# Clear the old graph
drop_query = "CALL gds.graph.drop('ds_graph', false)"


# Project a full graph (cannot filter here as it is schema based)
all_graph_query = """
CALL gds.graph.project(
    'ds_graph', 
    {
        Complaint: {
          properties: ['corp_community_id']   // project a property so we can use it in the filtering
        }
    },
    {
        SIMILAR: {
            type: 'SIMILAR', 
            orientation: 'NATURAL',
            properties: 'similarity_score'
        }
    }
)
"""

# Filter by Complaint properties
filtering_query = """
CALL gds.graph.filter(
  $sub,
  'ds_graph',
  $nodeFilter,
  'true'
)
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
"""

# Run a PageRank analysis
pagerank_query = """
CALL gds.pageRank.write(
  $sub,
  {
    relationshipWeightProperty: 'similarity_score',
    writeProperty: 'pageRankWithinCommunity'
  }
)
YIELD nodePropertiesWritten
RETURN nodePropertiesWritten
"""

drop_sub_query = "CALL gds.graph.drop($sub, false)"

with driver.session() as session:
    session.run(drop_query).consume()
    session.run(all_graph_query).consume()

    for id in corp_community_ids:
    
        cid = id
        sub = f"ds_graph_comm_{cid}"
        node_filter = f"n:Complaint AND n.corp_community_id = {cid}"

        # Create community subgraph
        stats = session.run(filtering_query, sub=sub, nodeFilter=node_filter).single()
        summary = session.run(pagerank_query, sub=sub).single()

        session.run(drop_sub_query, sub=sub).consume()


One example company

In [19]:
# Find the community_id of the Company node

query = """
MATCH (corp: Company)
WHERE toUpper(corp.name) CONTAINS "CAPITAL ONE"
RETURN corp.name, corp.community_id
"""

with driver.session() as session:
    corp = my_neo4j_run_query_pandas(query)

In [20]:
corp

,corp.name,corp.community_id
0,CAPITAL ONE FINANCIAL CORPORATION,37


In [21]:

query = """
MATCH (c:Complaint {corp_community_id: 21})
WHERE c.pageRankWithinCommunity IS NOT NULL
RETURN
  id(c) AS complaintId,
  c.pageRankWithinCommunity AS pageRankWithinCommunity,
  c.consumer_complaint_narrative AS narrative
ORDER BY c.pageRankWithinCommunity DESC
LIMIT 1
"""

with driver.session() as session:
    page = my_neo4j_run_query_pandas(query)


In [22]:
page

,complaintId,pageRankWithinCommunity,narrative


Loop for all communities

In [23]:
all_page_query = """

// For loop (for cid in communityIds)
UNWIND $communityIds AS cid

// Companies mentioned by complaints in the community
// Use call as a subquery
CALL {
    WITH cid
    MATCH (c:Complaint {corp_community_id: cid})-[:COMPLAIN_TO]->(corp:Company)
    // Returns a list of unique company names
    RETURN collect(DISTINCT corp.name) AS companies
}

// Top 3 complaints by PageRank within this community
CALL {
    WITH cid
    MATCH (c:Complaint {corp_community_id: cid})
    WHERE c.pageRankWithinCommunity IS NOT NULL
    WITH c
    // ORDER should come before collect (collect does not preserve order)
    ORDER BY c.pageRankWithinCommunity DESC
    LIMIT 3
    // Returns a list of map (a list of dict in python)
    RETURN collect({
    complaintId: id(c),
    pageRankWithinCommunity: c.pageRankWithinCommunity,
    narrative: c.consumer_complaint_narrative
    }) AS top3Complaints
}

RETURN cid AS communityId, companies, top3Complaints
ORDER BY communityId;

"""

with driver.session() as session:
    # Neo4j driver does not accept Python set object
    all_page_df = my_neo4j_run_query_pandas(all_page_query, communityIds=list(corp_community_ids))


In [24]:
# Results (example)
all_page_df.iloc[[4, 5,6]]

,communityId,companies,top3Complaints
4,19,[Fig Tech Inc.],"[{'complaintId': 110, 'narrative': 'I am submitting this complaint to document my attempt to resolve an alleged debt with Fig Loans. I am proposing a settlement of 50 % of the alleged balance of {$120.00}, amounting to {$61.00}, with the condition that upon receipt of this payment, Fig Loans agrees to delete the associated account from all credit reporting agencies. I am seeking confirmation from Fig Loans regarding their willingness to accept this settlement offer and to remove the negative tradeline from my credit reports upon payment.', 'pageRankWithinCommunity': 0.15000000000000002}]"
5,22,[WELLS FARGO & COMPANY],"[{'complaintId': 129, 'narrative': 'I am a Wells Fargo credit card holder. I recently requested a refund from , a travel agency, which was approved and issued on their end. Wells Fargo notified me that the refund was posted to my credit card account ; however, after carefully reviewing my transaction history, I do not see the refund reflected there. I have contacted Wells Fargo customer service multiple times in an effort to resolve this issue. Unfortunately, each time Ive called, Ive been treated rudely and have not received any clear or consistent explanation regarding the status of my refund. At this point, I am requesting that this matter be properly investigated and resolved. I would appreciate clear communication and documentation confirming the status of the refund and when I can expect to see it reflected in my account.', 'pageRankWithinCommunity': 0.15000000000000002}, {'complaintId': 340, 'narrative': 'always says they investigate when they dont they do not report the correct information. Been trying and they come back as no change or update with the same thing. tired of dealing with them.', 'pageRankWithinCommunity': 0.15000000000000002}]"
6,23,"[Portfolio Recovery Associates, LLC]","[{'complaintId': 144, 'narrative': 'ACCOUNT : I am filing this formal complaint against a company that is currently furnishing information to the credit reporting agencies regarding an alleged debt associated with my name and credit profile. I previously submitted a written request to this company demanding validation of the debt, including : A copy of the original contract bearing my signature Documentation establishing the chain of custody or assignment Any purchase agreement related to the alleged debt As of todays date, the company has failed to respond to my request. Under the Fair Debt Collection Practices Act ( FDCPA ), specifically 15 U.S. Code 1692g ( b ), I have the right to dispute the validity of a debt and to request proper documentation. Until the debt is validated, all collection activity and credit reporting must cease. Additionally, under the Fair Credit Reporting Act ( FCRA ), 15 U.S. Code 1681s-2, furnishers of credit information are required to provide accurate and verifiable data to the credit bureaus. Reporting a debt without providing supporting documentation upon request is a violation of federal law and causes unjust damage to my credit profile.', 'pageRankWithinCommunity': 0.15000000000000002}]"


In [25]:
# Timestamp (End)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

end_time = time.time()
elapsed_seconds = end_time - start_time

# Convert elapsed seconds to minutes and seconds
minutes = int(elapsed_seconds // 60)
seconds = elapsed_seconds % 60

print(f"Program elapsed time: {minutes} minutes and {seconds:.2f} seconds")

2026-01-08_22:23:58
Program elapsed time: 0 minutes and 0.64 seconds
